# Simple Campus Menu Agent

This notebook creates a simple agent that can answer questions about today's campus menu.
It uses the `get_menu` tool from `getmenus.py` to look up data from the database.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import display, Markdown
from loguru import logger

# Import the get_menu tool and helper functions from getmenus.py
from getmenus import get_menu, get_today, setup_logging

setup_logging()
load_dotenv()
print("Environment loaded.")

In [ ]:
logger.info(f"Today's date (Seoul time): {get_today()}")

## Create the Language Model

Connect to Azure OpenAI using credentials stored in `.env`.

In [ ]:
# Create the language model using Azure OpenAI credentials from the .env file
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

logger.info("LLM ready.")


## Build the Agent

Create the agent with the `get_menu` tool and a helpful system prompt.

In [ ]:
today = get_today()

# The system prompt tells the agent how to behave and what today's date is
SYSTEM_PROMPT = (
    f"You are a helpful campus menu assistant. Today's date is {today} (Asia/Seoul). "
    "Use the get_menu tool to fetch menu data for the requested date. "
    "Translate all Korean names (universities, restaurants, dishes) to English. "
    "Group the answer by university, then by restaurant. "
    "Include meal type, dish names, serving time, and price when available."
)

# Build the agent with the get_menu tool attached
agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_menu]
)

logger.info("Agent ready.")

## Ask the Agent a Question

Change the `question` variable below and run the cell to get an answer.

In [ ]:
# Change this question to ask anything about the menu
question = "What is on the menu today?"

In [ ]:
logger.info("Question: {!r}", question)

# Run the agent with our question
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Extract the final text reply from the agent's response messages
answer = ""
for message in reversed(result.get("messages", [])):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        answer = content
        break

logger.info("Answer: {} chars", len(answer))

# Display the answer nicely formatted in the notebook
display(Markdown(answer))